In [ ]:
# One-cell Colab: Runtime > Change runtime type > T4 GPU, then run.
import codecs, os, selectors, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive

BRANCH = 'feature/hada-mamba-ssm-v2'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
REPO_PATH = Path('/content/tcformer-mamba-ssm-v2-bcic2a')
SUBJECT_IDS = list(range(1, 10))
MAX_EPOCHS = 125
BATCH_SIZE = 48
USE_RA = True
USE_IM_TTA = True  # Transductive: adapts on unlabeled target-test EEG.
IM_TTA_STEPS = 5
RUN_TAG = f'ra-{int(USE_RA)}_imtta-{IM_TTA_STEPS if USE_IM_TTA else 0}'

if Path('/content/drive/MyDrive').is_dir():
    DRIVE_ROOT = Path('/content/drive')
else:
    mountpoint = Path('/content/drive')
    if mountpoint.exists() and any(mountpoint.iterdir()):
        mountpoint = Path('/content/google_drive')
    drive.mount(str(mountpoint))
    DRIVE_ROOT = mountpoint
MNE_DATA = DRIVE_ROOT / 'MyDrive/datasets/BCICIV2a_MNE'
OUTPUT_ROOT = DRIVE_ROOT / 'MyDrive/TCFormer-results'
RESULT_ARCHIVE = OUTPUT_ROOT / f'mamba_ssm_v2_bcic2a_loso_s01_s09_ep{MAX_EPOCHS}_{RUN_TAG}_results'
TRAIN_LOG = OUTPUT_ROOT / f'mamba_ssm_v2_bcic2a_loso_s01_s09_ep{MAX_EPOCHS}_{RUN_TAG}.log'

def run(command, cwd=None, env=None, stream=False, log_path=None):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    if not stream:
        subprocess.run(command, cwd=str(cwd) if cwd else None, env=env, check=True)
        return
    process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    assert process.stdout is not None
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    selector = selectors.DefaultSelector(); selector.register(process.stdout, selectors.EVENT_READ)
    started_at = last_heartbeat = time.monotonic()
    with open(log_path, 'w', encoding='utf-8') as log_file:
        while selector.get_map():
            for key, _ in selector.select(timeout=1):
                chunk = os.read(key.fileobj.fileno(), 4096)
                if not chunk:
                    selector.unregister(key.fileobj); continue
                output = decoder.decode(chunk)
                print(output, end='', flush=True); log_file.write(output); log_file.flush()
            now = time.monotonic()
            if process.poll() is None and now - last_heartbeat >= 30:
                heartbeat = f'[Colab heartbeat] training is running | elapsed={(now-started_at)/60:.1f}m\n'
                print(heartbeat, end='', flush=True); log_file.write(heartbeat); log_file.flush(); last_heartbeat = now
        remaining = decoder.decode(b'', final=True)
        if remaining:
            print(remaining, end='', flush=True); log_file.write(remaining)
    selector.close()
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Command failed with exit code {return_code}. Full log: {log_path}')

run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if (REPO_PATH / '.git').is_dir():
    run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_PATH)
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH)
    run(['git', 'checkout', BRANCH], cwd=REPO_PATH)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)

UV = shutil.which('uv') or 'uv'
run([UV, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.7.1', 'torchvision==0.22.1', '--index-url', 'https://download.pytorch.org/whl/cu126'])
run([UV, 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)
run([UV, 'pip', 'install', '--python', PYTHON, 'packaging', 'ninja', 'wheel', 'setuptools'])
run([UV, 'pip', 'install', '--python', PYTHON, '--no-build-isolation', '-r', 'requirements-mamba-v2.txt'], cwd=REPO_PATH)
smoke = "import torch, transformers, mamba_ssm, causal_conv1d; from causal_conv1d.cpp_functions import causal_conv1d_fwd_function; assert callable(causal_conv1d_fwd_function); from transformers.generation import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput; from mamba_ssm import Mamba2; assert transformers.__version__ == '4.44.2'; assert torch.cuda.is_available(); x=torch.randn(2,32,48,device='cuda',requires_grad=True); m=Mamba2(d_model=48,d_state=64,d_conv=4,expand=2,d_ssm=64,headdim=8).cuda(); y=m(x); y.sum().backward(); print('mamba-ssm:',mamba_ssm.__version__,'causal-conv1d:',causal_conv1d.__version__,'transformers:',transformers.__version__,'CUDA smoke:',tuple(y.shape),'GPU:',torch.cuda.get_device_name(0))"
run([PYTHON, '-c', smoke], stream=True, log_path=TRAIN_LOG)

override = f"import yaml; from pathlib import Path; p=Path('configs/hada_tcformer.yaml'); c=yaml.safe_load(p.read_text()); c['subject_ids']={SUBJECT_IDS!r}; c['max_epochs_loso']={MAX_EPOCHS}; pre=c['preprocessing']['bcic2a']; pre['batch_size']={BATCH_SIZE}; pre['riemannian_alignment']={USE_RA!r}; pre.setdefault('model_overrides', {{}})['im_tta_steps']={IM_TTA_STEPS if USE_IM_TTA else 0}; p.write_text(yaml.safe_dump(c, sort_keys=False))"
run([PYTHON, '-c', override], cwd=REPO_PATH)
MNE_DATA.mkdir(parents=True, exist_ok=True); OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
environment = os.environ.copy()
environment.update({'PYTHONUNBUFFERED':'1','PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True','MPLBACKEND':'Agg','MNE_DATA':str(MNE_DATA),'MNE_DATASETS_BNCI_PATH':str(MNE_DATA)})
print(f'===== OFFICIAL MAMBA V2 | BCI IV-2A LOSO S01-S09 | BS {BATCH_SIZE} | {MAX_EPOCHS} EPOCHS =====', flush=True)
print(f'Options: RA={USE_RA} | IM-TTA steps={IM_TTA_STEPS if USE_IM_TTA else 0}', flush=True)
print('Live log:', TRAIN_LOG, flush=True)
run([PYTHON, '-u', 'train_pipeline.py', '--model', 'hada_tcformer', '--dataset', 'bcic2a', '--loso', '--gpu_id', '0'], cwd=REPO_PATH, env=environment, stream=True, log_path=TRAIN_LOG)
archive = shutil.make_archive(str(RESULT_ARCHIVE), 'zip', root_dir=REPO_PATH, base_dir='results')
print('Full S01-S09 LOSO complete. Results:', archive, flush=True)
